In [3]:
import cv2
import numpy as np
import random
import os

# ============================================
# توابع اصلی بازی
# ============================================

def split_image_into_tiles(image_path, grid_size=4):
    """بارگذاری تصویر و تقسیم آن به تکه‌های مساوی"""
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f"تصویر در مسیر {image_path} پیدا نشد!")
    
    h, w, _ = img.shape
    tile_h, tile_w = h // grid_size, w // grid_size
    
    tiles = []
    for i in range(grid_size):
        for j in range(grid_size):
            tile = img[i*tile_h:(i+1)*tile_h, j*tile_w:(j+1)*tile_w]
            tiles.append(tile)
    
    # تکه آخر را خالی می‌کنیم (جای خالی)
    tiles[-1] = np.zeros((tile_h, tile_w, 3), dtype=np.uint8)
    return tiles, tile_h, tile_w, img


def shuffle_tiles(tiles, grid_size, empty_idx, moves=300):
    """به هم زدن تکه‌ها با حرکت‌های تصادفی"""
    idx = empty_idx
    for _ in range(moves):
        row, col = idx // grid_size, idx % grid_size
        neighbors = []
        if row > 0: neighbors.append(idx - grid_size)
        if row < grid_size - 1: neighbors.append(idx + grid_size)
        if col > 0: neighbors.append(idx - 1)
        if col < grid_size - 1: neighbors.append(idx + 1)
        
        neighbor = random.choice(neighbors)
        # جابه‌جایی
        tiles[idx], tiles[neighbor] = tiles[neighbor], tiles[idx]
        idx = neighbor
    return idx


def is_adjacent_to_empty(clicked_idx, empty_idx, grid_size):
    """بررسی می‌کند که آیا تکه کلیک شده کنار جای خالی است"""
    row1, col1 = clicked_idx // grid_size, clicked_idx % grid_size
    row2, col2 = empty_idx // grid_size, empty_idx % grid_size
    return abs(row1 - row2) + abs(col1 - col2) == 1


def check_win(tiles, original_tiles, grid_size):
    """بررسی می‌کند که آیا پازل کامل شده است"""
    for i in range(grid_size * grid_size):
        if i == grid_size * grid_size - 1:
            # تکه آخر باید خالی باشد
            if not np.all(tiles[i] == 0):
                return False
        else:
            # تکه‌های دیگر باید با حالت اولیه برابر باشند
            if not np.array_equal(tiles[i], original_tiles[i]):
                return False
    return True


def render_board(tiles, grid_size, tile_h, tile_w, is_win=False):
    """ساخت تصویر نهایی تخته برای نمایش"""
    rows = []
    for i in range(grid_size):
        row_tiles = tiles[i*grid_size:(i+1)*grid_size]
        rows.append(cv2.hconcat(row_tiles))
    board = cv2.vconcat(rows)
    
    if is_win:
        # اضافه کردن پیام پیروزی
        cv2.putText(board, "🎉 YOU WIN! 🎉", (50, 50), 
                    cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 255, 0), 4)
        cv2.putText(board, "Press 'r' to restart or 'q' to quit", 
                    (50, 100), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    
    return board


# ============================================
# کلاس مدیریت بازی
# ============================================

class PuzzleGame:
    def __init__(self, image_path, grid_size=4):
        self.grid_size = grid_size
        self.image_path = image_path
        self.window_name = "Puzzle Game"
        self.moves_count = 0
        self.is_win = False
        
        # بارگذاری و آماده‌سازی
        self.tiles, self.tile_h, self.tile_w, self.original_img = split_image_into_tiles(
            image_path, grid_size
        )
        
        # ذخیره حالت اولیه برای بررسی برنده شدن
        self.original_tiles = self.tiles.copy()
        
        # جای خالی در آخرین ایندکس
        self.empty_index = grid_size * grid_size - 1
        
        # به هم زدن تکه‌ها
        self.empty_index = shuffle_tiles(self.tiles, grid_size, self.empty_index, 300)
        
        # ایجاد پنجره و تنظیم کالبک
        cv2.namedWindow(self.window_name)
        cv2.setMouseCallback(self.window_name, self.mouse_callback)
        
        # نمایش اولیه
        self.update_display()
        
        print(f"✅ بازی شروع شد! ({grid_size}x{grid_size})")
        print("🖱️ روی تکه‌های کنار جای خالی کلیک کن تا جابه‌جا شوند.")
        print("⌨️ کلیدهای کنترل: [q] خروج  [r] شروع مجدد")
    
    def mouse_callback(self, event, x, y, flags, param):
        """مدیریت کلیک ماوس"""
        if event == cv2.EVENT_LBUTTONDOWN and not self.is_win:
            col = x // self.tile_w
            row = y // self.tile_h
            
            # بررسی اینکه کلیک داخل محدوده باشد
            if row < self.grid_size and col < self.grid_size:
                clicked_idx = row * self.grid_size + col
                
                # اگر تکه کنار جای خالی بود، جابه‌جا کن
                if is_adjacent_to_empty(clicked_idx, self.empty_index, self.grid_size):
                    self.swap_tiles(clicked_idx)
                    self.moves_count += 1
                    
                    # بررسی برنده شدن
                    if check_win(self.tiles, self.original_tiles, self.grid_size):
                        self.is_win = True
                        print(f"🎉 تبریک! پازل را در {self.moves_count} حرکت حل کردی!")
                    
                    # به‌روزرسانی نمایش
                    self.update_display()
    
    def swap_tiles(self, clicked_idx):
        """جابه‌جایی تکه کلیک شده با جای خالی"""
        self.tiles[self.empty_index], self.tiles[clicked_idx] = \
            self.tiles[clicked_idx], self.tiles[self.empty_index]
        self.empty_index = clicked_idx
    
    def update_display(self):
        """به‌روزرسانی پنجره"""
        board = render_board(
            self.tiles, self.grid_size, self.tile_h, self.tile_w, self.is_win
        )
        cv2.imshow(self.window_name, board)
        
        # نمایش تعداد حرکت‌ها در ترمینال
        if not self.is_win:
            print(f"حرکت: {self.moves_count}", end="\r")
    
    def restart(self):
        """شروع مجدد بازی"""
        print("\n🔄 شروع مجدد...")
        self.tiles = self.original_tiles.copy()
        self.empty_index = self.grid_size * self.grid_size - 1
        self.empty_index = shuffle_tiles(self.tiles, self.grid_size, self.empty_index, 300)
        self.moves_count = 0
        self.is_win = False
        self.update_display()
        print("✅ بازی دوباره شروع شد!")
    
    def run(self):
        """حلقه اصلی بازی"""
        while True:
            key = cv2.waitKey(1) & 0xFF
            
            if key == ord('q'):
                break
            elif key == ord('r'):
                self.restart()
        
        cv2.destroyAllWindows()
        print("👋 بازی بسته شد.")


# ============================================
# اجرای بازی
# ============================================

if __name__ == "__main__":
    # ======== تنظیمات ========
    GRID_SIZE = 3  # 3 یا 4 یا 5
  
    # مسیر تصویر را وارد کن
    IMAGE_PATH = "ele.png"  
    
   
    # ======== اجرا ========
    try:
        game = PuzzleGame(IMAGE_PATH, GRID_SIZE)
        game.run()
    except Exception as e:
        print(f"❌ خطا: {e}")
        print("نکات رفع خطا:")
        print("1. مسیر تصویر را بررسی کن")
        print("2. مطمئن شو که فایل تصویر وجود دارد")
        print("3. اگر در Jupyter هستی، از مسیر کامل استفاده کن")

✅ بازی شروع شد! (3x3)
🖱️ روی تکه‌های کنار جای خالی کلیک کن تا جابه‌جا شوند.
⌨️ کلیدهای کنترل: [q] خروج  [r] شروع مجدد
حرکت: 21

KeyboardInterrupt: 